# TPC Event 3D Viewer

Interactive visualization of the point-cloud CSV produced by `tpcanalysis-main/run_mini.cpp`.

- **event_id**: click the box, then ↑/↓ to step ±1 or type a number directly.
- **voxel / min_count**: voxel-coincidence filter. `voxel=0` disables it (default). Try `voxel=1 mm, min_count=10` for a gentle clean-up, `min_count=20` for aggressive ghost rejection.
- **max pts**: safety cap on rendered markers.
- **n lines**: 0 disables line fitting; >0 runs iterative BFGS and overlays up to N lines. Fitting takes a few seconds per event.
- **fit thresh**: per-point distance (mm) for a point to count as an inlier of a fitted line.

In [1]:
import sys
sys.path.insert(0, '.')
from viewer3d import load_points, plot_event
import ipywidgets as widgets

CSV_PATH = '../raw_run_files/CoBo_2026-04-06_0001_points.csv'
df = load_points(CSV_PATH)
events = sorted(df['event_id'].unique().tolist())
print(f'Loaded {len(df)} points across {len(events)} events (range: {events[0]}..{events[-1]})')

Loaded 13640132 points across 142 events (range: 0..141)


In [2]:
event = widgets.BoundedIntText(
    value=events[0], min=events[0], max=events[-1], step=1,
    description='event_id',
)
voxel = widgets.FloatSlider(
    value=0.0, min=0.0, max=5.0, step=0.5, description='voxel [mm]',
)
min_count = widgets.IntSlider(
    value=1, min=1, max=500, step=1, description='min_count',
)
max_points = widgets.IntSlider(
    value=20000, min=1000, max=200000, step=1000, description='max pts',
)
n_lines = widgets.IntSlider(
    value=0, min=0, max=6, step=1, description='n lines',
)
fit_thresh = widgets.FloatSlider(
    value=3.0, min=0.5, max=10.0, step=0.5, description='fit thresh',
)
out = widgets.Output()

def render(change=None):
    out.clear_output(wait=True)
    with out:
        try:
            fig = plot_event(
                df, event.value,
                voxel_size_mm=voxel.value,
                min_count=min_count.value,
                max_points=max_points.value,
                n_lines=n_lines.value,
                fit_threshold_mm=fit_thresh.value,
            )
            fig.show()
        except ValueError as e:
            print(e)

for w in (event, voxel, min_count, max_points, n_lines, fit_thresh):
    w.observe(render, names='value')

widgets.VBox([
    widgets.HBox([event, voxel]),
    widgets.HBox([min_count, max_points]),
    widgets.HBox([n_lines, fit_thresh]),
    out,
])

In [3]:
render()